In [ ]:
from google.colab import userdata
GIT_TOKEN = userdata.get('GITHUB_TOKEN')

In [ ]:
# Install deployment/export tooling
!pip install ethos-u-vela onnx onnxsim onnx2tf tensorflow pyyaml


In [ ]:
# Clone and install this project, then clone/install LibreYOLO as the detector package.
import os

GIT_USERNAME = "bencejdanko"
GIT_REPO = f"github.com/{GIT_USERNAME}/fomo-people-counting.git"

!git clone https://{GIT_USERNAME}:{GIT_TOKEN}@{GIT_REPO}
%cd /content/fomo-people-counting
!git pull
!pip install -e .
!test -d libreyolo || git clone https://github.com/LibreYOLO/libreyolo.git
!pip install -e ./libreyolo


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from pathlib import Path

from yolo_core.pipeline import YoloDetectionAnnotator, export_hf_dataset_specs_to_yolo
from yolo_core.model import train_libreyolo_model, export_libreyolo_onnx
from yolo_core.quantization import (
    convert_onnx_to_tflite_with_onnx2tf,
    compile_tflite_with_vela,
)


In [ ]:
# 1. Dataset Setup
repo_ids = [
    "bdanko/overhead-people-rgb",
    "bdanko/loaf_resolution_512",
]

class_names = ["person"]
category_id_to_class_id = {1: 0}
loaf_center_crop_size = 192
center_crop_sizes = {
    "bdanko/loaf_resolution_512": loaf_center_crop_size,
}

train_datasets = []
val_datasets = []
test_datasets = []

for repo_id in repo_ids:
    dataset = load_dataset(repo_id)
    train_split = dataset["train"]
    split = train_split.train_test_split(test_size=0.2, seed=15179996)
    val_test = split["test"].train_test_split(test_size=0.5, seed=15179996)

    train_datasets.append((repo_id, split["train"]))
    val_datasets.append((repo_id, val_test["train"]))
    test_datasets.append((repo_id, val_test["test"]))

annotator = YoloDetectionAnnotator(
    class_names=class_names,
    category_id_to_class_id=category_id_to_class_id,
    merge_all_categories=False,
)


In [ ]:
# 2. Export Hugging Face datasets to YOLO format
yolo_data_dir = Path("./yolo_dataset")
data_yaml, dataset_stats = export_hf_dataset_specs_to_yolo(
    {
        "train": train_datasets,
        "validation": val_datasets,
        "test": test_datasets,
    },
    yolo_data_dir,
    annotator,
    center_crop_sizes=center_crop_sizes,
    overwrite=True,
)

print(data_yaml)
dataset_stats


In [ ]:
# 3. Sanity visualization
import random
from PIL import Image

sample_images = sorted((yolo_data_dir / "train" / "images").glob("*.jpg"))[:8]
fig, axes = plt.subplots(1, len(sample_images), figsize=(4 * len(sample_images), 4))
if len(sample_images) == 1:
    axes = [axes]

for ax, image_path in zip(axes, sample_images):
    image = Image.open(image_path).convert("RGB")
    ax.imshow(image)
    ax.axis("off")
    label_path = yolo_data_dir / "train" / "labels" / f"{image_path.stem}.txt"
    width, height = image.size
    for line in label_path.read_text().splitlines():
        class_id, x, y, w, h = map(float, line.split())
        x0 = (x - w / 2) * width
        y0 = (y - h / 2) * height
        rect = plt.Rectangle((x0, y0), w * width, h * height, fill=False, color="lime", linewidth=2)
        ax.add_patch(rect)

plt.show()


In [ ]:
# 4. Train LibreYOLO nano detector
model_ref = "yolox-n"
size = "n"
imgsz = 416
batch_size = 16
epochs = 100

model, train_results = train_libreyolo_model(
    data_yaml=data_yaml,
    model_ref=model_ref,
    size=size,
    epochs=epochs,
    batch=batch_size,
    imgsz=imgsz,
    project="runs/yolo_train",
    name="libreyolo_nano_people",
    device="auto",
    workers=4,
    lr0=0.01,
    relu6=False,
)

train_results


In [ ]:
# 5. Export ONNX
onnx_path = export_libreyolo_onnx(
    model,
    imgsz=imgsz,
    opset=13,
    output_dir="runs/yolo_train/libreyolo_nano_people/exports",
)
onnx_path


In [ ]:
# 6. Convert to int8 TFLite
# onnx2tf support for a given detector graph can vary by opset and installed TensorFlow version.
tflite_path = convert_onnx_to_tflite_with_onnx2tf(
    onnx_path,
    output_dir="tflite_output",
    quantize_int8=True,
)
tflite_path


In [ ]:
# 7. Compile with Vela for Ethos-U55
vela_output_dir = compile_tflite_with_vela(
    tflite_path,
    output_dir="vela_output",
    vela_config="configs/default_vela.ini",
    accelerator_config="ethos-u55-256",
)
vela_output_dir
